In [1]:
import os
import time
import string

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates

from tqdm import tqdm

# Font
from matplotlib import font_manager
font_path = "/workspace/KENTECH/fonts/"
font_list = os.listdir(font_path)
for font_file in font_list:
    try:
        font_manager.fontManager.addfont(font_path + font_file)
    except:
        raise Exception(f"Cannot Load {font_path+font_file}")

plt.rcParams["figure.dpi"] = 300

# Handling the DB

In [9]:
import sys
sys.path.append( '/workspace/KISTI_DB_Manager/' )

from KISTI_DB_Manager import manage, preview, processing
import importlib as imp
imp.reload(manage), imp.reload(preview), imp.reload(processing)

plt.rcParams["figure.dpi"] = 300

# WOS

In [3]:
path = '/workspace/share/Data-HDD/Web of Science/Web of Science (2024)/annual/'
flist = sorted([x for x in os.listdir(path) if x[0] != '.'])

In [10]:
f = flist[1]

def WOS_extract_jsons(path, f):
    import zipfile
    import gzip
    import json
    
    with zipfile.ZipFile(path+f, 'r') as zip_ref:
        # List all contents of the ZIP file, assuming there's only one gz file inside
        gz_file_name = [f for f in zip_ref.namelist() if f.endswith('.gz')][0]
        
        # Extract the gz file content
        with zip_ref.open(gz_file_name) as gz_file:
            # Use gzip on the extracted file-like object
            with gzip.open(gz_file, 'rt') as json_file:  # 'rt' mode for text mode reading
                content = json_file.read()
                # extracted_dict = json.load(json_file)
    xml = processing.read_a_xml(content)
    jsons = xml['records.REC'] # Real Record part
    return jsons

jsons = WOS_extract_jsons(path, f)

In [11]:
index_key = 'UID'
except_keys = ['dynamic_data'] # exception part for unstructured json branches

df, df_subs, excepted = processing.extract_data_from_jsons(jsons[:], index_key, except_keys)

100%|████████████████████████████████████| 51166/51166 [01:52<00:00, 454.64it/s]


In [12]:
df_subs.keys()

dict_keys(['static_data__summary__titles__title', 'static_data__fullrecord_metadata__category_info__subjects__subject'])

# Excepted branch

In [17]:
excepted['dynamic_data'][5]

{'cluster_related': {'identifiers': {'identifier': [{'@type': 'accession_no',
     '@value': 'XR374'},
    {'@type': 'issn', '@value': '0010-8901'}]}},
 'UID': 'WOS:A1945XR37400001'}

In [ ]:
df_desc = preview.get_Table_Description(data_config, params)

## Consider Excepted part
- accumulate the date
    - single > multiple
    - couverage checing
    - coloring
    - optimizion

In [ ]:
data_config = {
    'PATH': 'Data_Sample/',
    'SEP': ',',
    'file_name': 'file_name', # Dummy init value
    'file_type': 'csv', # Dummy init value
    'table_name': 'table_name', # Dummy init value for Exporting
    'out_path': '../Data/SQL/', # Update as needed
    'Conv_DATETIME': False,
}

In [ ]:
def data_archiving(df, df_sub)

In [8]:
excepted

{'dynamic_data': [{'cluster_related': {'identifiers': {'identifier': [{'@type': 'accession_no',
       '@value': 'V44TK'},
      {'@type': 'issn', '@value': '0360-9081'}]}}},
  {'cluster_related': {'identifiers': {'identifier': {'@type': 'accession_no',
      '@value': 'YA862'}}},
   'citation_related': {'citation_topics': {'subj-group': {'subject': [{'@content-id': '1',
        '@content-type': 'macro',
        '#text': 'Clinical & Life Sciences'},
       {'@content-id': '1.34',
        '@content-type': 'meso',
        '#text': 'Orthopedics'},
       {'@content-id': '1.34.800',
        '@content-type': 'micro',
        '#text': 'Intramedullary Nailing'}]}},
    'SDG': {'sdg_category': '03 Good Health and Well-being'}}},
  {'cluster_related': {'identifiers': {'identifier': [{'@type': 'accession_no',
       '@value': 'YE675'},
      {'@type': 'xref_doi', '@value': '10.1016/S0366-0869(45)80005-9'}]}},
   'citation_related': {'citation_topics': {'subj-group': {'subject': [{'@content-id': 

In [20]:
PATH = '../Data/Funding/KR_NTIS/'
SEP = '\t'
Port = 0 # Port for DB with host
CHARACTER_SET = 'utf8mb4'
COLLATE = 'utf8mb4_unicode_520_ci'

params = dict(Extra_ratio=1.5, 
              Min_Year=1900, 
              Max_Year=2100, 
              unique_ratio_th=.5, 
              freq_ratio_th=1e-3)

db_config = {
    'host': 'localhost',  # Update as needed
    'user': 'user',       # Update as needed
    'password': '1234',       # Update as needed
    'database': 'KR_NTIS_2023_raw'  # Update as needed
}

data_config = {
    'PATH': PATH,
    'SEP': SEP,
    'file_name': 'file_name', # Dummy init value
    'file_type': 'csv', # Dummy init value
    'table_name': 'table_name', # Dummy init value for Exporting
    'out_path': '../Data/SQL/', # Update as needed
    'Conv_DATETIME': False,
}

In [ ]:
manage.init_MySQL()
try:
    manage.drop_DB(db_config['database'], db_config)
except:
    pass
manage.create_DB(db_config['database'], CHARACTER_SET, COLLATE, db_config)


# Generate the Tabular File list
flist = sorted([x for x in os.listdir(PATH) if 'raw.ftr' in x])
for f in tqdm(flist[:]):
    data_config = preview.update_data_config(f, data_config)
    df_desc = preview.get_Table_Description(data_config, params)
    
    # Generate and execute CREATE TABLE SQL
    manage.create_table(data_config, db_config)
    manage.fill_table_from_file(data_config, db_config)
    manage.set_index(db_config, data_config)
    manage.optimize_table(db_config, data_config)

 * Starting MySQL database server mysqld
   ...done.
Failed to drop database `KR_NTIS_2023_raw`. Error: (1049, "Unknown database 'KR_NTIS_2023_raw'")
Database `KR_NTIS_2023_raw` created successfully.


  0%|                                                    | 0/16 [00:00<?, ?it/s]

Generate the Description file for table `1_Projects_raw`
Table `1_Projects_raw` created successfully.
Data inserted into table `1_Projects_raw` successfully.
Set Index the 과제고유번호 on `1_Projects_raw` successfully.


In [ ]:
manage.backup_database_subprocess(db_config, data_config)

In [81]:
_df_subs.keys()

dict_keys(['static_data__summary__titles__title', 'static_data__summary__names__name', 'static_data__fullrecord_metadata__references__reference', 'static_data__fullrecord_metadata__category_info__subjects__subject', 'dynamic_data__citation_related__citation_topics__subj-group__subject'])